# Design Spaces 

## Setup

### Import Packages and Directories

In [1]:
import flowtangent as ft
import jax
import jax.numpy as jnp
import numpy as np

from pathlib import Path

from flowtangent.models.kernels import RBFKernel, Matern52Kernel
from flowtangent.models.surrogates import SVGPSurrogate
from flowtangent.models.manifolds import LinearPCA, Passthrough, ScalingManifold
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

import jax.numpy as jnp

In [2]:
import string
from functools import lru_cache

@lru_cache(maxsize=2048)
def surf_lookup(name: str):
    safe_name = name.replace("-", "_").replace(".", "_")
    if safe_name and safe_name[0] in string.digits:
        safe_name = f"af_{safe_name}"

    af = getattr(ft.airfoils, safe_name)

    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(batch: dict):
    names = batch["foil_name"]
    
    # 1. Strip Xarray metadata if it slipped through
    if hasattr(names, "values"):
        names = names.values
        
    # 2. Force it into a flat, pure NumPy array
    names = np.asarray(names).flatten()
    
    # 3. Now iteration yields pure strings (or numpy.str_), which str() handles perfectly
    surfaces = [surf_lookup(str(name)) for name in names]
    
    batch["y_af"] = jnp.stack(surfaces)
    return batch

### Import Dataset

In [3]:
data_dir = Path("/home/jordan/dev/datasets/mUIUC_5D.zarr")
data_dir.mkdir(parents=True, exist_ok=True)

In [4]:
from huggingface_hub import hf_hub_download
import zipfile

zip_path = hf_hub_download(repo_id="eden-uiuc/mUIUC-5D", repo_type="dataset", filename="mUIUC_5D.zarr.zip")

if not (data_dir / 'zarr.json').exists():
    print("Extracting mUIUC 5D Dataset")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Extraction Complete.")
else:
    print ("Stored dataset detected.")

print(f"Dataset ready at: {data_dir}")

Stored dataset detected.
Dataset ready at: /home/jordan/dev/datasets/mUIUC_5D.zarr


In [4]:
XFD_raw = ft.utils.data.XArrayDataset(data_dir).to_tabular(["run", "alpha"])
XFD = XFD_raw.dropna(["alpha"])
XFD = XFD.transform_variable(name2surf)
XFD._proxy._ds.load()

Dropping NaNs from dataset:   0%|          | 0/104 [00:00<?, ?it/s]

<xarray.Dataset> Size: 654MB
Dimensions:     (run: 358400, bend_var: 24, cond_var: 4, alpha: 42, polar_var: 9)
Coordinates:
  * bend_var    (bend_var) <U18 2kB 'Area' 'Slen' ... 'skin_Y_Ixx/t(Y-Yc)' 'J/t'
  * cond_var    (cond_var) <U10 160B 'flap_angle' 'flap_x/c' 'Re' 'Mach'
  * polar_var   (polar_var) <U7 252B 'alpha' 'CL' 'CD' ... 'Top_Itr' 'Bot_Itr'
Dimensions without coordinates: run, alpha
Data variables:
    bend_data   (run, bend_var) float32 34MB 0.0463 2.051 0.3786 ... 0.04929 nan
    conditions  (run, cond_var) float32 6MB -1.81 0.7673 ... 1.978e+05 0.5435
    foil_name   (run) <U50 72MB '2032c' '2032c' '2032c' ... 'ys930' 'ys930'
    polar_data  (run, alpha, polar_var) float32 542MB 0.0 0.5962 ... 5.937 86.99

In [5]:
unique_names = [str(n) for n in set(np.asarray(XFD_raw.foil_name))][1:]
surfs = [jnp.atleast_2d(surf_lookup(n)) for n in unique_names]
X_SURF = jnp.concatenate(surfs)

### Set up Parameters

In [6]:
# Inputs
y_af = Parameter(name="y_af", is_transform=True, transform_sources=("foil_name",))
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Parameter(name="alpha")

# Outputs
CL = Parameter(name="CL", is_output=True)
CD = Parameter(name="CD", is_output=True)
CM = Parameter(name="CM", is_output=True)

theta = (y_af, Mach, Re, flap_loc, flap_angle, alpha, CL, CD, CM)

### Set up Manifold

In [7]:
geofold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
geofold = geofold.fit(X_SURF)

machfold    = ScalingManifold(method="softplus")
refold      = ScalingManifold(method="logarithmic")
hingefold   = ScalingManifold(method="algebraic_bounded", bounds_min=0.6, bounds_max=0.85)
deltafold   = ScalingManifold(method="linear", scale_factors=10)
alphafold   = ScalingManifold(method="linear", scale_factors=15)

mfold = geofold + machfold + refold + hingefold + deltafold + alphafold

### Setup Design Space

In [8]:
batch_size = 8192
beta = 0.5 * batch_size / len(XFD)

mu_scl =  (XFD.CL.mean().item(), XFD.CD.mean().item(), XFD.CM.mean().item())
sig_scl = (XFD.CL.std().item(), XFD.CD.std().item(), XFD.CM.std().item())

krnl = Matern52Kernel(input_dim=21)
surr = SVGPSurrogate(
    jax.random.key(42),
    input_dim=21,
    output_dim=3,
    kernel=krnl,
    num_inducing=256,
    complexity_weight=beta,
    jitter=1e-5,
    # scale_mean=mu_scl,
    # scale_std=sig_scl
)

airspace = ActiveSpace(
    dataset=XFD,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
)

In [9]:
airspace = airspace.fit_models(batch_size=batch_size, surrogate_epochs=1)

Training Neural Model (notebook mode):   0%|          | 0/1 [00:00<?, ?it/s]

FigureWidget({
    'data': [{'line': {'color': '#2563eb'},
              'mode': 'lines',
              'name': 'Loss',
              'type': 'scatter',
              'uid': 'f7fd0b1f-9c00-41c2-bd46-dfb88e74ebff',
              'y': []}],
    'layout': {'template': '...', 'title': {'text': 'Training Convergence'}, 'yaxis': {'type': 'log'}}
})

Epoch 1 Batches:   0%|          | 0/1585 [00:00<?, ?it/s]

In [10]:
def generate_diagnostics(model, test_x, test_y, n_bins=21):
    # 1. Base Predictions & Metrics
    preds = jax.vmap(model.surrogate.predict)(test_x)
    residuals = preds.means - test_y
    absolute_errors = jnp.abs(residuals)

    safe_y = jnp.where(jnp.abs(test_y) < 1e-4, 1e-4, jnp.abs(test_y))
    
    pred_std = jnp.sqrt(preds.variances)
    z_scores = residuals / pred_std

    metrics = {
        metr: {
            "MAE": jnp.mean(absolute_errors[:, i]).item(),
            "RMSE": jnp.sqrt(jnp.mean(residuals[:, i]**2)).item(),
            "Z_Score_Variance": jnp.var(z_scores[:, i]).item() 
        }
        for i, metr in enumerate(["CL", "CD", "CM"])
    }

    # 2. Vectorized Binning
    m_dist = jnp.linalg.norm(test_x[:, :16], axis=1)
    
    bins_dict = {
        "Geometry": jnp.linspace(0., jnp.max(m_dist), n_bins),
        "Mach": jnp.linspace(0.0, 0.6, n_bins),
        "Re": jnp.logspace(np.log10(50_000), np.log10(3_000_000), n_bins),
        "Pivot": jnp.linspace(0.6, 0.85, n_bins),
        "Angle": jnp.linspace(-10, 15, n_bins),
        "Alpha": jnp.linspace(-5, 15, n_bins)
    }

    test_x_hat = jax.vmap(model.reconstruct)(test_x)
    features_dict = {
        "Geometry": m_dist,
        "Mach": test_x_hat["Mach"].flatten(),
        "Re": test_x_hat["Re"].flatten(),
        "Pivot": test_x_hat["flap_x/c"].flatten(),
        "Angle": test_x_hat["flap_angle"].flatten(),
        "Alpha": test_x_hat["alpha"].flatten(),
    }
    
    param_names = list(bins_dict.keys())
    features = jnp.stack([features_dict[p] for p in param_names]) # (6_params, N_points)
    bins_stack = jnp.stack([bins_dict[p] for p in param_names])   # (6_params, n_bins)
    
    # Digitize all features at once. Indices 1 to (n_bins-1) correspond to our defined intervals.
    idxs = jax.vmap(lambda f, b: jnp.digitize(f, b, right=True))(features, bins_stack)
    
    # Create boolean mask: (6_params, N_points, n_bins + 1)
    n_indices = n_bins + 1
    masks = idxs[..., None] == jnp.arange(n_indices)
    
    # 3. Tensor Aggregation for Binned Means
    sq_frac_errs = ((residuals / safe_y) ** 2).T  # (3_metrics, N_points)
    
    # Sum errors across points in each bin: (3_metrics, 6_params, n_indices)
    sum_sq_frac = jnp.einsum('mn,pnb->mpb', sq_frac_errs, masks)
    
    # Count points in each bin: (6_params, n_indices)
    counts = jnp.sum(masks, axis=1)
    
    # Divide by count (MSE), then square root (RMSE)
    mean_sq_frac = jnp.where(counts > 0, sum_sq_frac / counts, np.nan)
    rms_frac_errors = jnp.sqrt(mean_sq_frac)
    
    valid_rms_frac = rms_frac_errors[:, :, 1:n_bins]
    
    errors = {}
    for p_idx, param in enumerate(param_names):
        b = bins_dict[param]
        centers = (b[:-1] + b[1:]) / 2.0
        
        errors[param] = {
            "bins": b,
            "centers": centers,
            "means": { 
                metr: valid_rms_frac[m_idx, p_idx, :]
                for m_idx, metr in enumerate(["CL", "CD", "CM"])
            }
        }
        
    return preds, metrics, z_scores, errors

In [12]:
from plotly import graph_objects as go
from plotly.subplots import make_subplots

def plot_error_matrices(errors):
    param_names = list(errors.keys())
    metrics = ["CL", "CD", "CM"]
    
    fig = make_subplots(
        rows=2, cols=3, 
        subplot_titles=param_names,
        horizontal_spacing=0.08,
        vertical_spacing=0.15
    )
    
    for i, param in enumerate(param_names):
        row = (i // 3) + 1
        col = (i % 3) + 1
        
        centers = errors[param]["centers"]
        z_data = [errors[param]["means"][m] for m in metrics]
        
        fig.add_trace(
            go.Heatmap(
                z=z_data,
                x=centers,
                y=metrics,
                colorscale="Viridis", 
                zmin=0.0, 
                zmax=2.0,
                # Optional: set zmax=0.5 (50%) to prevent zero-crossing spikes from washing out the color scale
                showscale=(i == 0),
                colorbar=dict(
                    title="RMS % Error",
                    tickformat=".1%"  # Automatically formats 0.15 as 15.0%
                ) if i == 0 else None,
            ),
            row=row, col=col
        )
        
        if param == "Re":
            fig.update_xaxes(type="log", row=row, col=col)
            
    fig.update_layout(
        title_text="Surrogate RMS Percentage Error",
        height=600, 
        width=1000,
        plot_bgcolor='white'
    )
    
    return fig

In [23]:
# 1. Generate globally random indices
total_points = len(airspace.dataset)
test_indices = np.random.choice(total_points, size=8192, replace=False)

# Sort indices to ensure Zarr reads the chunks sequentially without thrashing
test_indices.sort()

# 2. Extract the batch instantly via the new vectorized fast-path
raw_batch = airspace.dataset[test_indices]

geo_x = raw_batch["y_af"]

# 3. Stack the inputs and targets
scalar_x = jnp.stack([
    raw_batch["Mach"], 
    raw_batch["Re"], 
    raw_batch["flap_x/c"], 
    raw_batch["flap_angle"], 
    raw_batch["alpha"]
], axis=-1)

raw_x = jnp.concatenate([geo_x, scalar_x], axis=-1)

test_y = jnp.stack([
    raw_batch["CL"], 
    raw_batch["CD"], 
    raw_batch["CM"]
], axis=-1)

# 4. Push through the manifold
test_x = jax.vmap(airspace.manifold.encode)(raw_x)

# 5. Generate matrices
preds, metrics, z_scores, errors = generate_diagnostics(airspace, test_x, test_y)



In [82]:
ZDL = iter(ft.utils.data.DataLoader(airspace.dataset, batch_size=8192, shuffle=True))
# test_x, test_y = next(ZDL)
# preds, metrics, z_scores, errors = generate_diagnostics(airspace, test_x, test_y)

In [24]:
plot_error_matrices(errors)

In [24]:
metrics = ["CL", "CD", "CM"]

for i, e in enumerate(evals):
    print(f"CL: Pred - {e.means[0]:.3f} ({e.variances[0]:.3f}); Target: [{XFD[i]["CL"]:.3f}]({(e.means[0]- XFD[i]["CL"])/XFD[i]["CL"] * 100 :.2f}%, {(e.means[0]- XFD[i]["CL"])/XFD[i]["CL"]/e.variances[0]:.2f})")
    print(f"CD: Pred - {e.means[1]:.3f} ({e.variances[1]:.3f}); Target: [{XFD[i]["CD"]:.3f}]({(e.means[1]- XFD[i]["CD"])/XFD[i]["CD"] * 100 :.2f}%, {(e.means[1]- XFD[i]["CD"])/XFD[i]["CD"]/e.variances[1]:.2f})")
    print(f"CM: Pred - {e.means[2]:.3f} ({e.variances[2]:.3f}); Target: [{XFD[i]["CM"]:.3f}]({(e.means[2]- XFD[i]["CM"])/XFD[i]["CM"] * 100 :.2f}%, {(e.means[2]- XFD[i]["CM"])/XFD[i]["CM"]/e.variances[2]:.2f})\n")

CL: Pred - 0.417 (0.056); Target: [0.596](-30.06%, -5.40)
CD: Pred - 0.043 (0.001); Target: [0.036](18.69%, 251.39)
CM: Pred - -0.097 (0.001); Target: [-0.131](-26.19%, -351.11)

CL: Pred - 0.563 (0.054); Target: [0.729](-22.78%, -4.19)
CD: Pred - 0.041 (0.001); Target: [0.033](25.50%, 351.28)
CM: Pred - -0.101 (0.001); Target: [-0.130](-22.48%, -308.65)

CL: Pred - 0.703 (0.052); Target: [0.841](-16.42%, -3.13)
CD: Pred - 0.040 (0.001); Target: [0.030](33.16%, 473.17)
CM: Pred - -0.104 (0.001); Target: [-0.125](-16.75%, -238.31)

CL: Pred - 0.833 (0.050); Target: [0.944](-11.79%, -2.36)
CD: Pred - 0.040 (0.001); Target: [0.034](16.92%, 253.69)
CM: Pred - -0.106 (0.001); Target: [-0.123](-13.54%, -202.37)

CL: Pred - 0.949 (0.047); Target: [1.021](-7.05%, -1.50)
CD: Pred - 0.041 (0.001); Target: [0.042](-1.97%, -31.36)
CM: Pred - -0.106 (0.001); Target: [-0.121](-12.29%, -195.27)



In [24]:
for i, e in enumerate(evals):
    print(f"CL: Pred - {e.means[0]:.3f} ({e.variances[0]:.3f}); Target: [{XFD[i]["CL"]:.3f}]({(e.means[0]- XFD[i]["CL"])/XFD[i]["CL"] * 100 :.2f}%, {(e.means[0]- XFD[i]["CL"])/XFD[i]["CL"]/e.variances[0]:.2f})")
    print(f"CD: Pred - {e.means[1]:.3f} ({e.variances[1]:.3f}); Target: [{XFD[i]["CD"]:.3f}]({(e.means[1]- XFD[i]["CD"])/XFD[i]["CD"] * 100 :.2f}%, {(e.means[1]- XFD[i]["CD"])/XFD[i]["CD"]/e.variances[1]:.2f})")
    print(f"CM: Pred - {e.means[2]:.3f} ({e.variances[2]:.3f}); Target: [{XFD[i]["CM"]:.3f}]({(e.means[2]- XFD[i]["CM"])/XFD[i]["CM"] * 100 :.2f}%, {(e.means[2]- XFD[i]["CM"])/XFD[i]["CM"]/e.variances[2]:.2f})\n")

CL: Pred - 0.382 (0.032); Target: [0.596](-35.99%, -11.32)
CD: Pred - 0.029 (0.001); Target: [0.036](-19.84%, -136.79)
CM: Pred - -0.112 (0.001); Target: [-0.131](-14.80%, -107.19)

CL: Pred - 0.492 (0.032); Target: [0.729](-32.49%, -10.25)
CD: Pred - 0.030 (0.001); Target: [0.033](-9.29%, -68.90)
CM: Pred - -0.113 (0.001); Target: [-0.130](-13.35%, -104.40)

CL: Pred - 0.600 (0.032); Target: [0.841](-28.65%, -9.04)
CD: Pred - 0.031 (0.001); Target: [0.030](2.52%, 18.86)
CM: Pred - -0.113 (0.001); Target: [-0.125](-9.86%, -77.93)

CL: Pred - 0.703 (0.032); Target: [0.944](-25.57%, -8.06)
CD: Pred - 0.032 (0.001); Target: [0.034](-4.72%, -34.72)
CM: Pred - -0.111 (0.001); Target: [-0.123](-9.21%, -71.52)

CL: Pred - 0.798 (0.032); Target: [1.021](-21.88%, -6.89)
CD: Pred - 0.035 (0.001); Target: [0.042](-16.20%, -117.03)
CM: Pred - -0.109 (0.001); Target: [-0.121](-10.60%, -80.63)



In [16]:
dae41_1 = int(np.where(XFD.foil_name == "dae41")[0][0])

In [20]:
jnp.linalg.norm(airspace.project(XFD[dae41_1])[:16])

Array(0.03270292, dtype=float64)